In [ ]:
from astropy.coordinates import SkyCoord
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import requests
import time
import os

FRITZ_API_KEY = os.environ.get('FRITZ_API_KEY')
if FRITZ_API_KEY is None:
    host = None
    headers = None
    raise ValueError("FRITZ_API_KEY is not set")
else:
    host = "https://fritz.science"
    headers = {'Authorization': f'token {FRITZ_API_KEY}'}

## Load Sources

In [ ]:
all_not_saved = pd.read_csv("../not_saved_sources.txt", sep="\t")
no_junk_not_saved = pd.read_csv("../not_saved_sources_no_junk.txt", sep="\t")

all_not_saved['in_junk'] = all_not_saved['ZTFID'].isin(no_junk_not_saved['ZTFID'])

source_coords = SkyCoord(all_not_saved['ra'], all_not_saved['dec'], unit="deg")
len(all_not_saved)

## Cross-match Helper

In [ ]:
def crossmatch_catalog(source_df, source_coords, catalog_df, catalog_coords,
                       col_mapping, prefix, radius_arcsec=1.0):
    """
    Cross-match sources against a reference catalog.

    For each source, finds the nearest catalog entry and stores match info
    if the separation is below `radius_arcsec`.

    Parameters
    ----------
    source_df : pd.DataFrame
        Source table (modified in place).
    source_coords : SkyCoord
        Sky coordinates of the sources.
    catalog_df : pd.DataFrame
        Reference catalog (must have a default 0-based integer index).
    catalog_coords : SkyCoord
        Sky coordinates of the catalog entries.
    col_mapping : dict
        {output_column_name: catalog_column_name} for columns to copy on match.
    prefix : str
        Prefix for the _match and _sep columns.
    radius_arcsec : float
        Maximum separation in arcseconds for a valid match.
    """
    source_df[f'{prefix}_match'] = False
    source_df[f'{prefix}_sep'] = np.nan
    for out_col in col_mapping:
        source_df[out_col] = None

    for i in tqdm(range(len(source_coords)), desc=prefix):
        seps = catalog_coords.separation(source_coords[i])
        best = np.argmin(seps)
        if seps[best].arcsec < radius_arcsec:
            source_df.at[i, f'{prefix}_match'] = True
            source_df.at[i, f'{prefix}_sep'] = seps[best].arcsec
            for out_col, cat_col in col_mapping.items():
                source_df.at[i, out_col] = catalog_df.iloc[best][cat_col]

    n = source_df[f'{prefix}_match'].sum()
    print(f"{prefix}: {n} matches out of {len(source_df)} sources")

## Milliquas Cross-match

In [ ]:
milliquas_path = Path("../Milliquas/milliquas.txt").resolve()

colspecs = [
    (0, 11), (12, 23), (25, 50), (51, 55), (56, 61), (62, 67),
    (68, 71), (72, 73), (74, 75), (76, 82), (83, 89), (90, 96),
    (97, 119), (120, 142), (143, 165), (166, 188),
]
names = [
    "RA", "DEC", "Name", "Type", "Rmag", "Bmag", "Comment",
    "R", "B", "Z", "Cite", "Zcite", "Xname", "Rname", "Lobe1", "Lobe2",
]

milliquas = pd.read_fwf(milliquas_path, colspecs=colspecs, names=names)
for c in ("Name", "Type", "Comment", "R", "B", "Cite", "Zcite", "Xname", "Rname", "Lobe1", "Lobe2"):
    milliquas[c] = milliquas[c].astype("string").str.strip()
for c in ("RA", "DEC", "Rmag", "Bmag", "Z"):
    milliquas[c] = pd.to_numeric(milliquas[c], errors="coerce")

milliquas = milliquas[['RA', 'DEC', 'Name', 'Type', 'Z']]
milliquas.head()

print(len(milliquas), "sources in Milliquas")

In [ ]:
milliquas_coords = SkyCoord(milliquas["RA"], milliquas["DEC"], unit="deg")

crossmatch_catalog(
    all_not_saved, source_coords,
    milliquas, milliquas_coords,
    col_mapping={'MQ_name': 'Name', 'MQ_z': 'Z', 'MQ_type': 'Type'},
    prefix='MQ',
)

## Chen+20 Periodic Variable Stars Cross-match

In [ ]:
chen20_colspecs = [
    (0, 22), (23, 29), (30, 39), (40, 49), (50, 61), (62, 68),
    (69, 74), (75, 88), (89, 95), (96, 102), (103, 116), (117, 130),
    (131, 135), (136, 140), (141, 147), (148, 154), (155, 160),
    (161, 166), (167, 173), (174, 180), (181, 186), (187, 192),
    (193, 201), (202, 210), (211, 216), (217, 222), (223, 228),
]

chen20_names = [
    'ID', 'Seq', 'RAdeg', 'DEdeg', 'Per', 'R21', 'phi21', 'T0',
    'gmag', 'rmag', 'Per_g', 'Per_r', 'Ng', 'Nr', 'R21_g', 'R21_r',
    'phi21_g', 'phi21_r', 'R2_g', 'R2_r', 'gAmp', 'rAmp', 'logFAP_g',
    'logFAP_r', 'Type', 'Dmin_g', 'Dmin_r',
]

chen20_df = pd.read_fwf(
    "../VarStars/table2.dat",
    colspecs=chen20_colspecs,
    names=chen20_names,
    comment='#',
    skip_blank_lines=True,
)

chen20_df = chen20_df[['ID', 'RAdeg', 'DEdeg', 'Per', 'Type']]
chen20_df.head()
print(len(chen20_df), "sources in Chen+20 catalog")

In [ ]:
chen20_coords = SkyCoord(chen20_df["RAdeg"], chen20_df["DEdeg"], unit="deg")

crossmatch_catalog(
    all_not_saved, source_coords,
    chen20_df, chen20_coords,
    col_mapping={'PVS_ID': 'ID', 'PVS_period': 'Per', 'PVS_type': 'Type'},
    prefix='PVS',
)


## Cataclysmic Variables Cross-match

In [ ]:
cv_df = pd.read_csv("../CVs/ZTF_CataclysmicVariables - CVs.csv")
cv_df = cv_df.dropna(subset=['ra', 'dec']).reset_index(drop=True)

cv_df = cv_df[['ZTF_id', 'ra', 'dec']]

cv_df.head()
print(len(cv_df), "sources in CVs")

In [ ]:
cv_coords = SkyCoord(cv_df["ra"], cv_df["dec"], unit="deg")

crossmatch_catalog(
    all_not_saved, source_coords,
    cv_df, cv_coords,
    col_mapping={'CV_ZTF_id': 'ZTF_id'},
    prefix='CV',
)

## Fritz Classifications

In [ ]:
types_to_select = [
    'AGN', 'QSO', 'Blazar', 'BL Lac', 'Cataclysmic', 'varstar',
    'Mira', 'LPV', 'Seyfert',
]

types_to_reject = [
    'Ia', 'Type II', 'Tidal Disruption Event', 'duplicate', 'bogus',
    'Ic', 'Ic-SLSN', 'roid', 'Ia-91bg', 'afterglow', 'Ia-CSM',
    'Solar System Object', 'IIb', 'Type I',
]


def get_fritz_classifications(ztfid):
    url = f"https://fritz.science/api/sources/{ztfid}/classifications"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return [clf['classification'] for clf in response.json()['data']]
    return []

In [ ]:
all_not_saved['Fritz_classifications'] = None

for idx in tqdm(range(len(all_not_saved)), desc="Fritz query"):
    ztfid = all_not_saved.at[idx, 'ZTFID']
    all_not_saved.at[idx, 'Fritz_classifications'] = get_fritz_classifications(ztfid)
    time.sleep(0.1)

In [ ]:
all_not_saved['Fritz_match'] = False

for idx in tqdm(range(len(all_not_saved)), desc="Fritz filter"):
    fritz_clfs = all_not_saved.at[idx, 'Fritz_classifications']

    good_type = False
    for clf in fritz_clfs:
        if clf in types_to_select:
            good_type = True
        elif clf in types_to_reject:
            good_type = False
            break

    all_not_saved.at[idx, 'Fritz_match'] = good_type

print(f"Fritz matches: {all_not_saved['Fritz_match'].sum()} / {len(all_not_saved)}")

## Add labels

Priority order: MQ -> CVs -> Fritz -> PVS

In [ ]:
print(len(all_not_saved[all_not_saved['MQ_match'] & all_not_saved['PVS_match']]), "MQ & PVS")
print(len(all_not_saved[all_not_saved['MQ_match'] & all_not_saved['CV_match']]), "MQ & CV")
print(len(all_not_saved[all_not_saved['MQ_match'] & all_not_saved['Fritz_match']]), "MQ & Fritz")
print(len(all_not_saved[all_not_saved['PVS_match'] & all_not_saved['CV_match']]), "PVS & CV")
print(len(all_not_saved[all_not_saved['PVS_match'] & all_not_saved['Fritz_match']]), "PVS & Fritz")
print(len(all_not_saved[all_not_saved['CV_match'] & all_not_saved['Fritz_match']]), "CV & Fritz")


In [ ]:
MQ_PVS_comatch = all_not_saved.loc[all_not_saved['MQ_match'] & all_not_saved['PVS_match'], ['ZTFID', 'MQ_type', 'PVS_type']]

MQ_PVS_comatch

# PVS often incorrectly labels AGN as SR 

In [ ]:
MQ_Fritz_comatch = all_not_saved.loc[all_not_saved['MQ_match'] & all_not_saved['Fritz_match'], ['ZTFID', 'MQ_type', 'Fritz_classifications']]

print(MQ_Fritz_comatch['MQ_type'].value_counts())
print(MQ_Fritz_comatch['Fritz_classifications'].value_counts())

# Overlapping matches agree

In [ ]:
PVS_CV_comatch = all_not_saved.loc[all_not_saved['PVS_match'] & all_not_saved['CV_match'], ['ZTFID', 'PVS_type', 'PVS_period']]

print(PVS_CV_comatch)

# Cataclysmic binary

In [ ]:
PVS_Fritz_comatch = all_not_saved.loc[(all_not_saved['PVS_match']) & (all_not_saved['Fritz_match']) & (~all_not_saved['MQ_match']), ['ZTFID', 'PVS_type', 'Fritz_classifications']]

print(PVS_Fritz_comatch)

# Similarly to MQ-PVS overlapping matches, PVS often incorrectly classifies AGN as SR

In [ ]:
CV_Fritz_comatch = all_not_saved.loc[(all_not_saved['CV_match']) & (all_not_saved['Fritz_match']), ['ZTFID', 'Fritz_classifications']]

CV_Fritz_comatch[CV_Fritz_comatch['Fritz_classifications'].str.join(",") != "Cataclysmic"]

# most are CVs
# ZTF18abdfkrf, ZTF18abbjkcd possibly AGN?
# ZTF18abvjznu unclear

In [65]:
all_not_saved['label'] = None

all_not_saved.loc[all_not_saved['MQ_match'], 'label'] = 'AGN'
all_not_saved.loc[all_not_saved['label'].isna() & all_not_saved['CV_match'], 'label'] = 'CV'

for idx in all_not_saved[all_not_saved['label'].isna()].index:
    fritz_classifications = all_not_saved.loc[idx, 'Fritz_classifications']
    is_AGN = any([agn_class in fritz_classifications for agn_class in ['AGN', 'QSO', 'BL Lac', 'Seyfert']])
    if is_AGN:
        all_not_saved.loc[idx, 'label'] = 'AGN'

    is_VarStar = any([varstar_class in fritz_classifications for varstar_class in ['varstar', 'LPV', "Mira"]])
    if is_VarStar:
        all_not_saved.loc[idx, 'label'] = 'VarStar'
    
    is_CV = any([cv_class in fritz_classifications for cv_class in ['Cataclysmic']])
    if is_CV:
        all_not_saved.loc[idx, 'label'] = 'CV'

all_not_saved.loc[all_not_saved['label'].isna() & all_not_saved['PVS_match'], 'label'] = 'VarStar'

mislabeled_AGN = ["ZTF18abdfkrf", "ZTF18abbjkcd"]
all_not_saved.loc[all_not_saved['ZTFID'].isin(mislabeled_AGN), 'label'] = 'AGN'


## Save Results

In [66]:
all_not_saved.to_csv("../not_saved_sources_crossmatched.txt", sep="\t", index=False)